# Зчитування та очищення даних 
* Зчитати завантажені текстові файли у pandas dataframe. Здійснити data cleaning: прибрати зайві стовпці, заповнити пропуски, видалити зайвий текст тощо.
* Додати стовпчики з назвою та індексом області.
* Реалізувати процедуру зміни індексів: замінити індекси так, щоб області індексувалася за українською абеткою.

In [1]:
import os
import pandas as pd

# Словник для заміни індексів на українську абетку
noaa_to_ukr = {
    1: 22, 2: 24, 3: 23, 4: 25, 5: 3, 6: 4, 7: 8, 8: 19, 9: 20, 10: 21,
    11: 9, 12: 26, 13: 10, 14: 11, 15: 12, 16: 13, 17: 14, 18: 15, 19: 16,
    20: 27, 21: 17, 22: 18, 23: 6, 24: 1, 25: 2, 26: 7, 27: 5
}

def create_vhi_dataframe(folder_path):
    all_data = []
    
    # 8 колонок
    headers = ['year', 'week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI', 'empty']
    
    for filename in os.listdir(folder_path):
        if filename.endswith(".csv"):
            file_path = os.path.join(folder_path, filename)
            province_id = int(filename.split('_')[2])
            
            # файл без заголовків 
            df = pd.read_csv(file_path, header=None, names=headers, skipinitialspace=True)
            
            # Видаляємо порожню 8-му колонку
            df = df.drop(columns=['empty'])
            
            # Видаляємо рядок, який був текстовим заголовком у файлі ('year', 'week' і т.д.)
            df = df[df['year'] != 'year']
            
            # Видаляємо пропуски
            df = df.dropna(subset=['year', 'VHI'])
            
            # Фільтруємо рядки від можливого сміття (наприклад HTML-тегів)
            df = df[~df['year'].astype(str).str.contains('<', na=False)]
            
            # Перетворюємо рік та VHI на правильний числовий формат
            df['year'] = df['year'].astype(int)
            df['VHI'] = pd.to_numeric(df['VHI'], errors='coerce')
            
            # Додаємо стовпчики з індексом області
            df['area_NOAA'] = province_id
            df['area_UKR'] = df['area_NOAA'].map(noaa_to_ukr)
            
            all_data.append(df)
            
    # Об'єднуємо всі таблиці в одну
    full_df = pd.concat(all_data, ignore_index=True)
    
    # Видаляємо рядки з некоректними значеннями VHI (-1.00)
    full_df = full_df[full_df['VHI'] != -1.00]
    
    return full_df

# Виклик функції та вивід перших 5 рядків
vhi_df = create_vhi_dataframe('vhi_data')
vhi_df.head()

,year,week,SMN,SMT,VCI,TCI,VHI,area_NOAA,area_UKR
0,1982,1,0.059,258.24,51.11,48.78,49.95,10,21
1,1982,2,0.063,261.53,55.89,38.20,47.04,10,21
2,1982,3,0.063,263.45,57.30,32.69,44.99,10,21
3,1982,4,0.061,265.10,53.96,28.62,41.29,10,21
4,1982,5,0.058,266.42,46.87,28.57,37.72,10,21


### Процедури формування вибірок (Частина 1)
Реалізувати процедури формування вибірок:
* Ряд ѴНІ для області за вказаний рік.
* Пошук екстремумів (min та max) для вказаних областей та років.

In [2]:
# Функція 1: Ряд VHI для області за вказаний рік
def get_vhi_for_year(df, province_id_ukr, year):
    # Фільтруємо за українським індексом області та роком
    result = df[(df['area_UKR'] == province_id_ukr) & (df['year'] == year)]
    # Повертаємо колонки тижня та VHI
    return result[['week', 'VHI']]

# Функція 2: Пошук екстремумів (min та max) для вказаної області та року
def get_vhi_extremes(df, province_id_ukr, year):
    data = df[(df['area_UKR'] == province_id_ukr) & (df['year'] == year)]
    vhi_min = data['VHI'].min()
    vhi_max = data['VHI'].max()
    return vhi_min, vhi_max

# --- Виклик функцій для перевірки ---
# Тестуємо на 1-й області (Вінницька) за 2000 рік

print("--- Ряд VHI для 1 області (Вінницька) за 2000 рік (перші 5 тижнів) ---")
vhi_series = get_vhi_for_year(vhi_df, 1, 2000)
print(vhi_series.head())

print("\n--- Екстремуми VHI для 1 області за 2000 рік ---")
min_vhi, max_vhi = get_vhi_extremes(vhi_df, 1, 2000)
print(f"Мінімум: {min_vhi}, Максимум: {max_vhi}")

--- Ряд VHI для 1 області (Вінницька) за 2000 рік (перші 5 тижнів) ---
      week    VHI
34476    1  24.22
34477    2  27.70
34478    3  30.68
34479    4  32.55
34480    5  34.73

--- Екстремуми VHI для 1 області за 2000 рік ---
Мінімум: 11.25, Максимум: 63.27


### Процедури формування вибірок (Частина 2)
Реалізувати процедури формування вибірок:
* Ряд ѴНІ за вказаний діапазон років для вказаних областей.
* Пошук екстремумів (min та max) для вказаних областей та років, середнього, медіани.

In [3]:
# Функція 3: Ряд VHI за вказаний діапазон років для вказаних областей
def get_vhi_for_years_and_provinces(df, provinces_ukr_list, start_year, end_year):
    # Фільтруємо за списком областей (isin) та діапазоном років
    result = df[(df['area_UKR'].isin(provinces_ukr_list)) & 
                (df['year'] >= start_year) & 
                (df['year'] <= end_year)]
    return result[['year', 'week', 'VHI', 'area_UKR']]

# Функція 4: Пошук екстремумів (min, max), середнього та медіани 
def get_vhi_statistics(df, provinces_ukr_list, years_list):
    # Фільтруємо за потрібними областями та роками
    data = df[(df['area_UKR'].isin(provinces_ukr_list)) & 
              (df['year'].isin(years_list))]
    
    stats = {
        'Мінімум': data['VHI'].min(),
        'Максимум': data['VHI'].max(),
        'Середнє': data['VHI'].mean(),
        'Медіана': data['VHI'].median()
    }
    return stats

# --- Виклик функцій для перевірки ---
print("--- VHI для Вінницької (1) та Київської (10) областей за 2010-2012 роки ---")
vhi_range = get_vhi_for_years_and_provinces(vhi_df, [1, 10], 2010, 2012)
print(vhi_range.head(10))

print("\n--- Статистика для Київської (10) області за 2015 та 2016 роки ---")
stats = get_vhi_statistics(vhi_df, [10], [2015, 2016])
for key, value in stats.items():
    print(f"{key}: {value:.2f}")

--- VHI для Вінницької (1) та Київської (10) областей за 2010-2012 роки ---
      year week    VHI  area_UKR
8164  2010    1  59.51        10
8165  2010    2  59.37        10
8166  2010    3  58.84        10
8167  2010    4  58.29        10
8168  2010    5  57.22        10
8169  2010    6  56.20        10
8170  2010    7  54.47        10
8171  2010    8  52.84        10
8172  2010    9  52.11        10
8173  2010   10  49.63        10

--- Статистика для Київської (10) області за 2015 та 2016 роки ---
Мінімум: 26.05
Максимум: 76.98
Середнє: 47.08
Медіана: 44.87
